# Home Credit model card analysis

This notebook adds validation metrics, a cited lending-economics sensitivity analysis, saved-model SHAP explanations, and simulated selection rates by gender and education. **The saved final model is unchanged.**

The user explicitly authorized three temporary CV fits to recover the original out-of-fold (OOF) predictions. Those fits have already completed in the separate `src/recover_model_card_oof.py` utility. **Running this notebook performs no training or calibration fitting.**

**Business threshold status: [NEEDS INPUT].** A threshold derived from external financial benchmarks is selected for analysis. It is not a final Home Credit lending recommendation; the required profit, loss, exposure, horizon, and label mappings are not established.

All tables and figures are exported to `reports/model_card/`. CSVs retain numeric precision; Markdown summaries and PNG/SVG figures are provided for model-card writing.

## 1. Reproducible inputs and validation provenance

Use `.venv/model-card/bin/python`, with SHAP 0.50.0 and scikit-learn 1.7.2. All learned preparation and encoding are loaded from the existing saved pipeline. The original fold definitions and random seed 42 were reused for OOF recovery. These are model-development results, not nested CV or a new independent holdout. The fold models use internal early stopping; the saved final model uses its existing fixed 200-iteration refit.

The checks below verify row/ID alignment, OOF provenance, original AUC reproduction, saved-model hashes, and that the saved pipeline reproduces every competition-test submission probability.

In [ ]:
from pathlib import Path
import os, sys, tempfile
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "home_credit_mpl"))
os.environ.setdefault("NUMBA_CACHE_DIR", str(Path(tempfile.gettempdir()) / "home_credit_numba"))
os.environ.setdefault("OMP_NUM_THREADS", "4")
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, Markdown, Image
from src.model_card_analysis import (
    load_context, economic_scenarios, analyze_thresholds, analyze_groups,
    explain_saved_model, write_report, TABLES, FIGURES, OUT, BENCHMARK_ID,
)
pd.set_option("display.max_columns", 16)
pd.set_option("display.precision", 6)
ctx = load_context()
display(pd.Series({
    "OOF rows": len(ctx["oof"]),
    "Test rows": len(ctx["test"]),
    "Temporary CV fits explicitly authorized": ctx["oof_provenance"]["fit_count"],
    "Mean CV AUC": ctx["oof_provenance"]["mean_cv_auc"],
    "Pooled OOF AUC": ctx["oof_provenance"]["oof_auc"],
    "Final model retrained": False,
}, name="Verified result").to_frame())

## 2. Transparent profit/loss framework

Let **G** be the net gain on an approved loan that repays, and **L** the net loss on an approved loan that defaults. With rejection assigned zero incremental payoff:

$$E[\Pi\mid p,\mathrm{approve}]=(1-p)G-pL,\qquad t=\frac{G}{G+L}.$$

Simulate approval if **p < t** and classify payment difficulty if **p ≥ t**. At exact break-even this analysis rejects. The current **0.50** cutoff remains a comparison, not an assumed optimum. Costs of rejected/reviewed applications, loan sizes, cash-flow timing, and other policy constraints require an expanded model when supplied.

The probabilities concern the competition's `TARGET=1` payment-difficulty event, whose late-payment X/Y definition is undisclosed. They cannot automatically be treated as probabilities of charge-off or economic default. The Federal Reserve's [expected-loss modeling framework](https://www.federalreserve.gov/publications/2025-june-supervisory-stress-test-methodology-descriptions-supervisory-models.htm) distinguishes probability of default, loss given default, and exposure at default. This analysis keeps their missing alignment explicit.

### Cited benchmarks and limits of transfer

**Profit proxy.** [OneMain, FY2025 earnings release, February 5, 2026](https://www.sec.gov/Archives/edgar/data/1584207/000158420726000004/exhibit991earningsreleaseo.htm), C&I segment and additional-metrics tables, reports interest income, interest expense, operating expense and average net receivables for 2024–2025. We derive `(interest income − interest expense − operating expense) / average net receivables`. This is a **portfolio-level annual pre-credit-loss, pretax margin proxy**, not measured profit conditional on repayment. Other revenue and credit provisions are excluded; the latter would double-count loss in this simplified payoff exercise.

**Recovery and LGD proxy.** [Distaso, Roccazzella and Vrins, July 25, 2023 working paper](https://www.crc.business-school.ed.ac.uk/sites/crc/files/2024-01/Business-Cycle-and-Realized-Losses-in-the-Consumer-Credit-Industry.pdf), Table 1, printed page 8, reports mean recovery of **47% for Financing** and **48% for Banking**, with 353,891 and 240,936 observations respectively. These Italian collection cases span 2007–2019 and have recovery mandates of at most 365 days. We calculate `1 − recovery` as a gross loss-share proxy; collection costs, discounting, and the exposure denominator require validation before calling it Home Credit net economic LGD.

For broader sensitivity, [Bank of Italy, December 2025, page 2](https://www.bancaditalia.it/pubblicazioni/note-stabilita/2025-0048/Note-di-stabilita-finanziaria-e-vigilanza-n-48-ENG.pdf?language_id=1) reports unsecured bad-loan recovery rates **28% in 2023** and **37% in 2024**. This mixed firm/household population includes public guarantees, which materially affect the increase. It is not a consumer-only match.

The benchmark uses 2025 margin plus Financing recovery, fixed from external evidence before inspecting OOF payoff results. **Combining these quantities on equal normalized exposure requires an unvalidated time/exposure bridge.** No real-currency total is claimed. The professor's **loss:profit = 7:1** case is included only as an illustrative comparison (`t=1/8=0.125`).

In [ ]:
scenarios = economic_scenarios(ctx)
display(pd.read_csv(TABLES / "sourced_margin_inputs.csv").drop(columns=["source_url", "source_location"]))
display(scenarios[["scenario", "gain_proxy", "loss_proxy", "recovery_rate", "threshold", "status"]])
display(Markdown(
    f"**Selected analysis benchmark:** `t = {ctx['gain']:.9f} / "
    f"({ctx['gain']:.9f} + {ctx['loss']:.2f}) = {ctx['selected_threshold']:.9f}`. "
    "**Final operational threshold: [NEEDS INPUT].**"
))

In [ ]:
display(pd.read_csv(TABLES / "business_inputs_needed.csv"))

## 3. Precision, recall, confusion matrices and threshold sensitivity

All classification metrics below use **labeled OOF probabilities**, pooled across the original three folds. The positive class is `TARGET=1`. No metric is calculated by scoring the final model on its own training rows.

TN and FN are simulated selections; FP and TP are predicted difficulties/rejections. Precision is undefined if there are no predicted positives; missing denominators are not reported as zero. The full CSV contains exact counts at every tested threshold. The economics output reports both label-assigned benchmark payoff `G×TN − L×FN` and score-implied expectation. Neither is measured monetary profit, and a high payoff on these development folds does not establish a deployable optimum.

In [ ]:
metrics = analyze_thresholds(ctx)
display(metrics[["threshold", "precision", "recall", "predicted_default_rate", "simulated_selection_rate", "selected_benchmark"]])
display(Image(filename=str(FIGURES / "threshold_tradeoffs.png")))

In [ ]:
comparison = metrics[metrics.threshold.isin([ctx["selected_threshold"], 0.125, 0.50])]
display(comparison[["threshold", "TN", "FP", "FN", "TP", "precision", "recall", "predicted_default_rate"]])
display(Image(filename=str(FIGURES / "confusion_matrix_comparison.png")))

In [ ]:
display(Image(filename=str(FIGURES / "profit_sensitivity.png")))
display(ctx["payoffs"][ctx["payoffs"].scenario.eq(BENCHMARK_ID)][[
    "threshold", "oof_label_based_payoff_per_applicant",
    "oof_model_implied_payoff_per_applicant", "test_model_implied_payoff_per_applicant"
]])

### Probability reliability diagnostic

The analytic threshold relies on probabilities being reliable for the relevant economic event. The OOF reliability plot compares mean predicted probability with observed payment-difficulty frequency in quantile bins. It fits no calibrator and does not resolve the label-to-economic-default mismatch. See [scikit-learn probability calibration](https://scikit-learn.org/stable/modules/calibration.html).

In [ ]:
display(pd.read_csv(TABLES / "oof_calibration_bins.csv"))
display(Image(filename=str(FIGURES / "oof_calibration.png")))

## 4. SHAP on the saved final model

Explain a reproducible **1,000-row sample of the competition test**, sampled without replacement with seed 42. Preprocessing only transforms the sample; it does not fit. [SHAP TreeExplainer](https://shap.readthedocs.io/en/latest/generated/shap.TreeExplainer.html) uses `tree_path_dependent` perturbation and the training-path counts already stored in the model.

Values are contributions to **log-odds**, not probability percentage points. Positive SHAP pushes predicted `TARGET=1` risk above the model baseline; negative SHAP pushes it below. We verify that the logistic transform of baseline plus summed SHAP reconstructs the saved-model probabilities.

Feature rankings are at the encoded/engineered predictor level. Direction is reported using disjoint high/low feature contrasts and Spearman association; `|rho| ≥ 0.2` is a transparent descriptive convention, not a significance test. Disagreeing or weak trends are labeled mixed. Correlated raw and engineered fields share importance. In particular, larger negative `DAYS_BIRTH` values mean younger applicants. Effects describe this model and sample, not causes.

In [ ]:
top_features = explain_saved_model(ctx)
display(top_features[["rank", "feature", "mean_abs_SHAP_log_odds", "direction", "spearman_feature_SHAP"]])
display(pd.Series(ctx["shap_validation"], name="SHAP verification").to_frame())

In [ ]:
display(Image(filename=str(FIGURES / "shap_summary.png")))
display(Image(filename=str(FIGURES / "shap_top_dependence.png")))

## 5. Simulated selection rates by gender and education

Use the sourced **analysis benchmark threshold** above and `p < t`. The main population is all 48,744 competition-test rows scored by the unchanged final model. A separate OOF population is also reported for the labeled development sample. The two must not be pooled.

Group on the **raw** `CODE_GENDER` and `NAME_EDUCATION_TYPE` fields so unknown `XNA` and missing values remain visible. A group with no rows gets an undefined rate, never a fabricated zero. Group counts and threshold sensitivity are exported. These are **simulated selection rates, not observed approvals**. The data represent issued loans, not the full applicant population. Differences are descriptive and do not establish fairness or legal compliance.

In [ ]:
selected_groups = analyze_groups(ctx)
for population in selected_groups.population.unique():
    display(Markdown(f"### {population}"))
    display(selected_groups[selected_groups.population.eq(population)][[
        "attribute", "group", "n", "simulated_selected", "simulated_selection_rate", "rate_status"
    ]])
display(Image(filename=str(FIGURES / "simulated_selection_by_group.png")))

## 6. Model-card handoff and integrity checks

The report below is generated from the calculated tables, with exact source locations and explicit missing inputs. It is accompanied by machine-readable CSVs, readable Markdown tables, PNG/SVG figures, OOF predictions and provenance, and the SHAP arrays/sample IDs. Final model/configuration checksums must be unchanged.

**No final business recommendation is made from hypothetical values or transferred financial proxies.** The professor's 7:1 comparison is illustrative only. A business recommendation requires the information in `business_inputs_needed.csv`.

In [ ]:
report = write_report(ctx)
display(Markdown(f"Saved findings: `{(OUT / 'model_card_findings.md').relative_to(ROOT)}`"))
display(pd.Series({
    "Saved final model unchanged": True,
    "Notebook fitting calls": 0,
    "Authorized temporary CV fits (separate utility)": 3,
    "Selected analysis benchmark": ctx["selected_threshold"],
    "Final business threshold": "[NEEDS INPUT]",
    "SHAP sample rows": ctx["shap_validation"]["sample_rows"],
    "Maximum SHAP probability reconstruction error": ctx["shap_validation"]["max_probability_reconstruction_error"],
}, name="Completed checks").to_frame())